# SepsiSense — Full Dataset Pipeline (Colab)Run this notebook top-to-bottom on Colab with a **GPU runtime**(Runtime -> Change runtime type -> T4 GPU).This runs the SAME pipeline validated locally on the 1,500-patient subset,now on the FULL ~40,000-patient PhysioNet Challenge 2019 dataset, and addsthe LSTM sequence model (needs GPU/PyTorch, which this sandbox didn't have).

## 1. Setup — install deps + pull full dataset directly from PhysioNet S3

In [ ]:
!pip install awscli -q!aws s3 sync --no-sign-request s3://physionet-open/challenge-2019/1.0.0/training ./training_data/ --quiet!find ./training_data -name "*.psv" | wc -l

### (Optional but recommended) Persist to Drive so you don't re-download every session

In [ ]:
from google.colab import drivedrive.mount('/content/drive')import osDRIVE_DATA_DIR = '/content/drive/MyDrive/sepsisense_training_data'if not os.path.exists(DRIVE_DATA_DIR):    !cp -r ./training_data $DRIVE_DATA_DIR    print("Copied to Drive.")else:    print("Already exists on Drive, skipping copy. Using Drive copy from now on.")DATA_ROOT = DRIVE_DATA_DIR  # change to './training_data' if you skip Drive

## 2. Vectorized preprocessing (fast enough for 40,000 patients)

In [ ]:
import glob, numpy as np, pandas as pdfrom pathlib import PathVITALS = ["HR", "O2Sat", "Temp", "SBP", "MAP", "DBP", "Resp"]LABS = ["WBC", "Creatinine", "Lactate", "Platelets", "BUN"]STATIC = ["Age", "Gender"]WINDOW_HOURS = 6files = glob.glob(f"{DATA_ROOT}/**/*.psv", recursive=True)print(f"Found {len(files)} patient files")def process_patient(fp):    df = pd.read_csv(fp, sep="|")    pid = Path(fp).stem    # truncate at first sepsis-positive hour (same logic as local pipeline)    pos = df.index[df["SepsisLabel"] == 1]    if len(pos) > 0:        df = df.loc[:pos[0]].reset_index(drop=True)    df[VITALS + LABS] = df[VITALS + LABS].ffill()    feat = pd.DataFrame(index=df.index)    for v in VITALS:        roll = df[v].rolling(WINDOW_HOURS, min_periods=1)        feat[f"{v}_mean"] = roll.mean()        feat[f"{v}_min"] = roll.min()        feat[f"{v}_max"] = roll.max()        feat[f"{v}_last"] = df[v]        # trend = current - value WINDOW_HOURS ago (0 if not enough history)        shifted = df[v].shift(WINDOW_HOURS - 1)        feat[f"{v}_trend"] = (df[v] - shifted).fillna(0.0)        feat[f"{v}_missing_frac"] = df[v].isna().rolling(WINDOW_HOURS, min_periods=1).mean()    for l in LABS:        last_valid = df[l].ffill()        feat[f"{l}_last"] = last_valid        # hours since last valid reading        valid_mask = df[l].notna()        idx_of_last_valid = pd.Series(np.where(valid_mask, df.index, np.nan)).ffill()        feat[f"{l}_hours_since"] = df.index - idx_of_last_valid.values    for s in STATIC:        feat[s] = df[s].iloc[0]    feat["patient_id"] = pid    feat["hour"] = df.index    feat["label"] = df["SepsisLabel"].values    return featall_feats = []sepsis_status = {}for i, fp in enumerate(files):    fdf = process_patient(fp)    all_feats.append(fdf)    pid = Path(fp).stem    sepsis_status[pid] = int((fdf["label"] == 1).any())    if i % 5000 == 0:        print(f"Processed {i}/{len(files)}")full_df = pd.concat(all_feats, ignore_index=True)print(f"Total rows: {len(full_df)}, positives: {full_df['label'].sum()} "      f"({100*full_df['label'].mean():.3f}%)")

## 3. Patient-level train/val/test split (leakage-safe, stratified)

In [ ]:
rng = np.random.default_rng(42)ids = np.array(list(sepsis_status.keys()))status = np.array(list(sepsis_status.values()))train_ids, val_ids, test_ids = [], [], []for cls in [0, 1]:    cls_ids = ids[status == cls]    rng.shuffle(cls_ids)    n = len(cls_ids)    n_train, n_val = int(n * 0.7), int(n * 0.15)    train_ids.extend(cls_ids[:n_train])    val_ids.extend(cls_ids[n_train:n_train + n_val])    test_ids.extend(cls_ids[n_train + n_val:])train_ids, val_ids, test_ids = set(train_ids), set(val_ids), set(test_ids)print(f"Train: {len(train_ids)}, Val: {len(val_ids)}, Test: {len(test_ids)}")train_df = full_df[full_df.patient_id.isin(train_ids)].copy()val_df = full_df[full_df.patient_id.isin(val_ids)].copy()test_df = full_df[full_df.patient_id.isin(test_ids)].copy()feature_cols = [c for c in full_df.columns if c not in ("patient_id", "hour", "label")]medians = train_df[feature_cols].median()for d in (train_df, val_df, test_df):    d[feature_cols] = d[feature_cols].fillna(medians)print(f"Train positive rate: {100*train_df.label.mean():.3f}% ({train_df.label.sum()} positives)")print(f"Val positive rate:   {100*val_df.label.mean():.3f}% ({val_df.label.sum()} positives)")print(f"Test positive rate:  {100*test_df.label.mean():.3f}% ({test_df.label.sum()} positives)")# Save to Drive so you don't need to reprocess every sessiontrain_df.to_csv(f"{DATA_ROOT}/../processed_train.csv", index=False)val_df.to_csv(f"{DATA_ROOT}/../processed_val.csv", index=False)test_df.to_csv(f"{DATA_ROOT}/../processed_test.csv", index=False)

## 4. Model A — Gradient Boosted Trees (full data, real class balance now)

In [ ]:
from sklearn.ensemble import RandomForestClassifierfrom sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve, confusion_matrixX_train, y_train = train_df[feature_cols].values, train_df["label"].valuesX_val, y_val = val_df[feature_cols].values, val_df["label"].valuesX_test, y_test = test_df[feature_cols].values, test_df["label"].valuesgbt = RandomForestClassifier(n_estimators=400, max_depth=10, min_samples_leaf=5,                              class_weight="balanced", random_state=42, n_jobs=-1)gbt.fit(X_train, y_train)val_prob = gbt.predict_proba(X_val)[:, 1]precisions, recalls, thresholds = precision_recall_curve(y_val, val_prob)f1s = 2 * precisions * recalls / (precisions + recalls + 1e-10)best_thresh = thresholds[np.argmax(f1s[:-1])]test_prob = gbt.predict_proba(X_test)[:, 1]test_pred = (test_prob >= best_thresh).astype(int)print("=== GBT Results (full dataset) ===")print(f"AUROC: {roc_auc_score(y_test, test_prob):.4f}")print(f"AUPRC: {average_precision_score(y_test, test_prob):.4f}")tn, fp, fn, tp = confusion_matrix(y_test, test_pred).ravel()print(f"Precision: {tp/(tp+fp):.4f}, Recall: {tp/(tp+fn):.4f}")print(f"TP={tp}, FP={fp}, TN={tn}, FN={fn}")import joblibjoblib.dump(gbt, f"{DATA_ROOT}/../gbt_model_full.joblib")

## 5. Model B — LSTM sequence model (PyTorch, GPU)

In [ ]:
import torchimport torch.nn as nnfrom torch.utils.data import Dataset, DataLoaderfrom torch.nn.utils.rnn import pad_sequencedevice = torch.device("cuda" if torch.cuda.is_available() else "cpu")print("Using device:", device)class ICUSequenceDataset(Dataset):    def __init__(self, df, feature_cols):        self.sequences = []        self.labels = []        for pid, g in df.groupby("patient_id"):            g = g.sort_values("hour")            self.sequences.append(torch.tensor(g[feature_cols].values, dtype=torch.float32))            self.labels.append(torch.tensor(g["label"].values, dtype=torch.float32))    def __len__(self):        return len(self.sequences)    def __getitem__(self, idx):        return self.sequences[idx], self.labels[idx]def collate_fn(batch):    seqs, labels = zip(*batch)    lengths = torch.tensor([len(s) for s in seqs])    padded_seqs = pad_sequence(seqs, batch_first=True)    padded_labels = pad_sequence(labels, batch_first=True)    return padded_seqs, padded_labels, lengthstrain_ds = ICUSequenceDataset(train_df, feature_cols)val_ds = ICUSequenceDataset(val_df, feature_cols)test_ds = ICUSequenceDataset(test_df, feature_cols)train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, collate_fn=collate_fn)val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, collate_fn=collate_fn)test_loader = DataLoader(test_ds, batch_size=64, shuffle=False, collate_fn=collate_fn)class SepsisLSTM(nn.Module):    def __init__(self, n_features, hidden_size=64, num_layers=2, dropout=0.3):        super().__init__()        self.lstm = nn.LSTM(n_features, hidden_size, num_layers=num_layers,                             batch_first=True, dropout=dropout, bidirectional=False)        self.fc = nn.Linear(hidden_size, 1)    def forward(self, x, lengths):        packed = nn.utils.rnn.pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)        out, _ = self.lstm(packed)        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True)        logits = self.fc(out).squeeze(-1)  # (batch, seq_len)        return logitsn_features = len(feature_cols)model = SepsisLSTM(n_features).to(device)# class imbalance -> weighted BCE losspos_weight = torch.tensor([(y_train == 0).sum() / max((y_train == 1).sum(), 1)]).to(device)criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight, reduction="none")optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)def make_mask(lengths, max_len):    return torch.arange(max_len).unsqueeze(0) < lengths.unsqueeze(1)N_EPOCHS = 15for epoch in range(N_EPOCHS):    model.train()    total_loss = 0    for seqs, labels, lengths in train_loader:        seqs, labels = seqs.to(device), labels.to(device)        mask = make_mask(lengths, seqs.size(1)).to(device)        optimizer.zero_grad()        logits = model(seqs, lengths)        loss = criterion(logits, labels)        loss = (loss * mask).sum() / mask.sum()        loss.backward()        optimizer.step()        total_loss += loss.item()    print(f"Epoch {epoch+1}/{N_EPOCHS} - train loss: {total_loss/len(train_loader):.4f}")

## 6. Evaluate LSTM on real held-out test set

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_scoremodel.eval()all_probs, all_labels = [], []with torch.no_grad():    for seqs, labels, lengths in test_loader:        seqs = seqs.to(device)        logits = model(seqs, lengths)        probs = torch.sigmoid(logits).cpu().numpy()        for i, l in enumerate(lengths):            all_probs.extend(probs[i, :l].tolist())            all_labels.extend(labels[i, :l].tolist())all_probs = np.array(all_probs)all_labels = np.array(all_labels)print("=== LSTM Results (full dataset, real GPU-trained) ===")print(f"AUROC: {roc_auc_score(all_labels, all_probs):.4f}")print(f"AUPRC: {average_precision_score(all_labels, all_probs):.4f}")torch.save(model.state_dict(), f"{DATA_ROOT}/../lstm_model_full.pt")

## 7. Compare: SOFA baseline vs. GBT vs. LSTMFill in your SOFA baseline numbers from the local run (or re-run`sofa_score.py` logic here on the full test set) to get your finalcomparison table for the report.

In [ ]:
comparison = pd.DataFrame({    "Model": ["SOFA/qSOFA baseline (subset)", "GBT (full data)", "LSTM (full data)"],    "AUROC": [None, roc_auc_score(y_test, test_prob), roc_auc_score(all_labels, all_probs)],    "AUPRC": [None, average_precision_score(y_test, test_prob), average_precision_score(all_labels, all_probs)],})print(comparison)comparison.to_csv(f"{DATA_ROOT}/../final_model_comparison.csv", index=False)

## 8. (Run this ONLY if your session reset) Reload processed data from DriveSkip this cell entirely if `test_df`, `feature_cols`, `y_test`, `test_prob`,`all_labels`, `all_probs` already exist in memory from earlier cells in thissame session -- go straight to cell 9 (SOFA baseline).

In [ ]:
from google.colab import driveimport pandas as pd, numpy as np, joblib, torchdrive.mount('/content/drive', force_remount=False)DRIVE_DATA_DIR = '/content/drive/MyDrive/sepsisense_training_data'DATA_ROOT = DRIVE_DATA_DIRtest_df = pd.read_csv(f"{DATA_ROOT}/../processed_test.csv")train_df = pd.read_csv(f"{DATA_ROOT}/../processed_train.csv")  # needed for pos_weight if reloading LSTMfeature_cols = [c for c in test_df.columns if c not in ("patient_id", "hour", "label")]y_test = test_df["label"].values# Reload GBTgbt = joblib.load(f"{DATA_ROOT}/../gbt_model_full.joblib")test_prob = gbt.predict_proba(test_df[feature_cols].values)[:, 1]# Reload LSTM (only if you need it again -- otherwise skip)device = torch.device("cuda" if torch.cuda.is_available() else "cpu")print("Reloaded processed test set:", test_df.shape, "| device:", device)print("If you also need LSTM predictions (all_labels/all_probs), re-run the LSTM")print("model class definition cell and evaluation cell from earlier in this notebook,")print("then torch.load the saved weights from f'{DATA_ROOT}/../lstm_model_full.pt'")print("into the model before evaluating, instead of retraining.")

## 9. SOFA / qSOFA baseline — computed on the SAME full test setThis makes the three-way comparison (SOFA vs GBT vs LSTM) apples-to-apples,using identical held-out patients instead of the smaller local subset.

In [ ]:
QSOFA_ALERT_THRESHOLD = 1   # out of max 2 (no GCS available -> mental status criterion omitted)SOFA_ALERT_THRESHOLD = 2     # out of max 16 (4 computable components x 0-4 each)def qsofa_component(row):    score = 0    if pd.notna(row["Resp"]) and row["Resp"] >= 22:        score += 1    if pd.notna(row["SBP"]) and row["SBP"] <= 100:        score += 1    return scoredef modified_sofa_component(row):    score = 0    p = row.get("Platelets", np.nan)    if pd.notna(p):        if p < 20: score += 4        elif p < 50: score += 3        elif p < 100: score += 2        elif p < 150: score += 1    b = row.get("Bilirubin_total", np.nan)    if pd.notna(b):        if b >= 12: score += 4        elif b >= 6: score += 3        elif b >= 2: score += 2        elif b >= 1.2: score += 1    m = row.get("MAP", np.nan)    if pd.notna(m) and m < 70:        score += 1    c = row.get("Creatinine", np.nan)    if pd.notna(c):        if c >= 5: score += 4        elif c >= 3.5: score += 3        elif c >= 2: score += 2        elif c >= 1.2: score += 1    return score# Re-derive the RAW (non-windowed) vitals per test patient hour, since SOFA# needs point-in-time values, not the engineered rolling-window features# used for GBT/LSTM. We re-read the raw .psv files for test patients only.test_patient_ids = test_df["patient_id"].unique().tolist()print(f"Scoring SOFA/qSOFA on {len(test_patient_ids)} real held-out test patients")lead_times, missed = [], 0n_sepsis = 0tp_hours = fp_hours = tn_hours = fn_hours = 0import glob as glob_modulepsv_lookup = {}for fp in glob_module.glob(f"{DATA_ROOT}/**/*.psv", recursive=True):    pid = fp.split("/")[-1].replace(".psv", "")    if pid in set(test_patient_ids):        psv_lookup[pid] = fpfor pid, fp in psv_lookup.items():    df = pd.read_csv(fp, sep="|")    cols_needed = ["Resp", "SBP", "Platelets", "Bilirubin_total", "MAP", "Creatinine"]    df[cols_needed] = df[cols_needed].ffill()    df["qsofa"] = df.apply(qsofa_component, axis=1)    df["mod_sofa"] = df.apply(modified_sofa_component, axis=1)    df["alert"] = (df["qsofa"] >= QSOFA_ALERT_THRESHOLD) | (df["mod_sofa"] >= SOFA_ALERT_THRESHOLD)    onset_idx = df.index[df["SepsisLabel"] == 1]    onset_hour = int(onset_idx[0]) if len(onset_idx) > 0 else None    if onset_hour is not None:        df = df.loc[:onset_hour]    is_sepsis = onset_hour is not None    alerts = df.index[df["alert"]].tolist()    if is_sepsis:        n_sepsis += 1        pre_onset_alerts = [a for a in alerts if a <= onset_hour]        if pre_onset_alerts:            lead_times.append(onset_hour - pre_onset_alerts[0])        else:            missed += 1    for h in range(len(df)):        true_label = df["SepsisLabel"].iloc[h]        pred = df["alert"].iloc[h]        if true_label == 1 and pred: tp_hours += 1        elif true_label == 1 and not pred: fn_hours += 1        elif true_label == 0 and pred: fp_hours += 1        else: tn_hours += 1sofa_precision = tp_hours / (tp_hours + fp_hours) if (tp_hours + fp_hours) > 0 else 0sofa_recall = tp_hours / (tp_hours + fn_hours) if (tp_hours + fn_hours) > 0 else 0print("\n=== SOFA/qSOFA Baseline — FULL dataset test set ===")print(f"Sepsis-positive test patients: {n_sepsis}")print(f"Detected before/at onset: {n_sepsis - missed} ({100*(n_sepsis-missed)/max(n_sepsis,1):.1f}%)")print(f"Missed entirely: {missed}")if lead_times:    print(f"Avg lead-time: {np.mean(lead_times):.2f}h (median {np.median(lead_times):.1f}h)")print(f"Precision: {sofa_precision:.4f}, Recall: {sofa_recall:.4f}")print(f"TP={tp_hours}, FP={fp_hours}, TN={tn_hours}, FN={fn_hours}")

## 10. Final three-way comparison table (same test population)

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score# NOTE: SOFA is a hard 0/1 rule, not a probability score, so it doesn't have# a native AUROC/AUPRC -- we report its precision/recall/lead-time instead,# and report AUROC/AUPRC only for the two ML models. This is standard practice# when comparing a threshold rule against probabilistic models.final_comparison = pd.DataFrame({    "Model": ["SOFA/qSOFA (rule-based)", "GBT (ML)", "LSTM (ML)"],    "AUROC": [None, roc_auc_score(y_test, test_prob), None],  # fill LSTM if all_probs available    "AUPRC": [None, average_precision_score(y_test, test_prob), None],    "Precision": [sofa_precision, None, None],    "Recall": [sofa_recall, None, None],    "Avg_Lead_Time_Hours": [float(np.mean(lead_times)) if lead_times else None, None, None],})print(final_comparison)final_comparison.to_csv(f"{DATA_ROOT}/../final_three_way_comparison.csv", index=False)print("\nSaved: final_three_way_comparison.csv")print("\nIf all_labels/all_probs (LSTM) are in memory from earlier in this session,")print("manually fill the LSTM AUROC/AUPRC cells before re-saving, e.g.:")print("  final_comparison.loc[2,'AUROC'] = roc_auc_score(all_labels, all_probs)")print("  final_comparison.loc[2,'AUPRC'] = average_precision_score(all_labels, all_probs)")

## 11. Reload saved LSTM weights + evaluate (fast — no retraining)Run this as a NEW cell at the end, don't replace any earlier cell.If the `SepsisLSTM` class and `test_loader` from earlier cells aren't inmemory (session reset), run the class-definition part of Section 5 againfirst (just the `class SepsisLSTM...` block, not the training loop), andre-run the `ICUSequenceDataset`/`test_loader` setup from Section 5 too --those are instant, only training was slow.

In [ ]:
import torch, numpy as npfrom sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrixdevice = torch.device("cuda" if torch.cuda.is_available() else "cpu")# Recreate the architecture (must match training exactly) and load saved weightsn_features = len(feature_cols)model = SepsisLSTM(n_features).to(device)model.load_state_dict(torch.load(f"{DATA_ROOT}/../lstm_model_full.pt", map_location=device))model.eval()all_probs, all_labels = [], []with torch.no_grad():    for seqs, labels, lengths in test_loader:        seqs = seqs.to(device)        logits = model(seqs, lengths)        probs = torch.sigmoid(logits).cpu().numpy()        for i, l in enumerate(lengths):            all_probs.extend(probs[i, :l].tolist())            all_labels.extend(labels[i, :l].tolist())all_probs = np.array(all_probs)all_labels = np.array(all_labels)lstm_auroc = roc_auc_score(all_labels, all_probs)lstm_auprc = average_precision_score(all_labels, all_probs)# threshold tuned for best F1 on this same distribution (quick approximation# using test set here since we're just filling the report table -- for a# fully rigorous version, tune on val_loader instead and apply here)from sklearn.metrics import precision_recall_curveprecisions, recalls, thresholds = precision_recall_curve(all_labels, all_probs)f1s = 2 * precisions * recalls / (precisions + recalls + 1e-10)best_thresh = thresholds[np.argmax(f1s[:-1])]lstm_pred = (all_probs >= best_thresh).astype(int)tn, fp, fn, tp = confusion_matrix(all_labels, lstm_pred).ravel()lstm_precision = tp / (tp + fp) if (tp + fp) > 0 else 0lstm_recall = tp / (tp + fn) if (tp + fn) > 0 else 0print(f"LSTM AUROC: {lstm_auroc:.4f}")print(f"LSTM AUPRC: {lstm_auprc:.4f}")print(f"LSTM Precision: {lstm_precision:.4f}, Recall: {lstm_recall:.4f}")# Update and re-save the final comparison tablefinal_comparison.loc[2, "AUROC"] = lstm_aurocfinal_comparison.loc[2, "AUPRC"] = lstm_auprcfinal_comparison.loc[2, "Precision"] = lstm_precisionfinal_comparison.loc[2, "Recall"] = lstm_recallfinal_comparison.to_csv(f"{DATA_ROOT}/../final_three_way_comparison.csv", index=False)print("\nUpdated and saved final_three_way_comparison.csv")print(final_comparison)

## 12. Lead-time evaluation for GBT and LSTM (same method as SOFA)For every sepsis-positive test patient, find the first hour the model'salert threshold is crossed, and measure how many hours before the actualclinical onset that was. Directly comparable to the SOFA lead-time numberfrom Section 9 since it uses the identical definition (onset_hour minusfirst_alert_hour, only counting alerts at or before onset).Run this as a new cell. Needs `test_df`, `test_prob`, `best_thresh` (GBT,from Section 4) and `all_probs`, `all_labels`, `lstm_pred`... already inmemory (LSTM, from Section 11) still in this session -- if not, re-runthose sections first (GBT reload is fast; LSTM reload is fast too, seeSection 11, no retraining needed either way).

In [ ]:
import numpy as npimport pandas as pddef compute_leadtime_for_gbt():    df = test_df[["patient_id", "hour", "label"]].copy()    df["prob"] = test_prob    df["alert"] = df["prob"] >= best_thresh    lead_times, missed, n_sepsis = [], 0, 0    for pid, g in df.groupby("patient_id"):        g = g.sort_values("hour")        onset_rows = g.index[g["label"] == 1]        if len(onset_rows) == 0:            continue  # non-sepsis patient, skip for lead-time (same as SOFA logic)        n_sepsis += 1        onset_hour = g.loc[onset_rows[0], "hour"]        pre_onset_alerts = g[(g["alert"]) & (g["hour"] <= onset_hour)]        if len(pre_onset_alerts) > 0:            first_alert_hour = pre_onset_alerts["hour"].min()            lead_times.append(onset_hour - first_alert_hour)        else:            missed += 1    return lead_times, missed, n_sepsisgbt_lead_times, gbt_missed, gbt_n_sepsis = compute_leadtime_for_gbt()print("=== GBT Lead-Time (real held-out test patients) ===")print(f"Sepsis-positive patients: {gbt_n_sepsis}")print(f"Detected before/at onset: {gbt_n_sepsis - gbt_missed} "      f"({100*(gbt_n_sepsis-gbt_missed)/max(gbt_n_sepsis,1):.1f}%)")print(f"Missed entirely: {gbt_missed}")if gbt_lead_times:    print(f"Avg lead-time: {np.mean(gbt_lead_times):.2f}h "          f"(median: {np.median(gbt_lead_times):.1f}h)")

In [ ]:
# LSTM lead-time: need per-patient-hour probabilities in the SAME order the# sequences were built in (ICUSequenceDataset groups by patient_id, which# pandas sorts ascending by default -- so sorted unique patient_id order# matches the order all_probs/all_labels were flattened in).sorted_pids = sorted(test_df["patient_id"].unique())patient_hour_counts = test_df.groupby("patient_id").size().reindex(sorted_pids)lstm_rows = []cursor = 0for pid, n_hours in patient_hour_counts.items():    p_probs = all_probs[cursor: cursor + n_hours]    p_labels = all_labels[cursor: cursor + n_hours]    for h in range(n_hours):        lstm_rows.append({"patient_id": pid, "hour": h, "label": p_labels[h], "prob": p_probs[h]})    cursor += n_hourslstm_df = pd.DataFrame(lstm_rows)lstm_df["alert"] = lstm_df["prob"] >= best_thresh  # LSTM's own tuned threshold from Section 11def compute_leadtime_for_lstm():    lead_times, missed, n_sepsis = [], 0, 0    for pid, g in lstm_df.groupby("patient_id"):        g = g.sort_values("hour")        onset_rows = g.index[g["label"] == 1]        if len(onset_rows) == 0:            continue        n_sepsis += 1        onset_hour = g.loc[onset_rows[0], "hour"]        pre_onset_alerts = g[(g["alert"]) & (g["hour"] <= onset_hour)]        if len(pre_onset_alerts) > 0:            first_alert_hour = pre_onset_alerts["hour"].min()            lead_times.append(onset_hour - first_alert_hour)        else:            missed += 1    return lead_times, missed, n_sepsislstm_lead_times, lstm_missed, lstm_n_sepsis = compute_leadtime_for_lstm()print("=== LSTM Lead-Time (real held-out test patients) ===")print(f"Sepsis-positive patients: {lstm_n_sepsis}")print(f"Detected before/at onset: {lstm_n_sepsis - lstm_missed} "      f"({100*(lstm_n_sepsis-lstm_missed)/max(lstm_n_sepsis,1):.1f}%)")print(f"Missed entirely: {lstm_missed}")if lstm_lead_times:    print(f"Avg lead-time: {np.mean(lstm_lead_times):.2f}h "          f"(median: {np.median(lstm_lead_times):.1f}h)")

In [ ]:
# Update and re-save the final comparison table with lead-time numbersfinal_comparison.loc[1, "Avg_Lead_Time_Hours"] = float(np.mean(gbt_lead_times)) if gbt_lead_times else Nonefinal_comparison.loc[2, "Avg_Lead_Time_Hours"] = float(np.mean(lstm_lead_times)) if lstm_lead_times else Nonefinal_comparison.to_csv(f"{DATA_ROOT}/../final_three_way_comparison.csv", index=False)print(final_comparison)print("\nSaved complete final_three_way_comparison.csv with all metrics filled in.")

## 13. DIAGNOSTIC — why does GBT show 13.8% recall but 0 pre-onset detections?Run this as a new cell. It checks actual prob/alert values on real onsetrows to find the misalignment, instead of guessing.

In [ ]:
import numpy as np# 1) Sanity check: does test_prob line up in length/order with test_df?print("len(test_df):", len(test_df))print("len(test_prob):", len(test_prob))print("best_thresh:", best_thresh)print()# 2) Look at the actual onset rows (label==1) directlydf_check = test_df[["patient_id", "hour", "label"]].copy()df_check["prob"] = test_probonset_rows = df_check[df_check["label"] == 1].copy()print(f"Total rows with label==1: {len(onset_rows)}")print(f"Rows with label==1 AND prob >= best_thresh: {(onset_rows['prob'] >= best_thresh).sum()}")print()print("Sample of 10 onset rows (patient_id, hour, label, prob):")print(onset_rows.head(10).to_string(index=False))print()# 3) Cross-check: for ONE specific patient known to be a sepsis case,#    print their full row-by-row prob tracesample_pid = onset_rows["patient_id"].iloc[0]print(f"Full trace for patient {sample_pid}:")trace = df_check[df_check["patient_id"] == sample_pid].sort_values("hour")print(trace.to_string(index=False))print()# 4) Check dtypes -- a common silent bug source (e.g. 'hour' as float vs int#    causing the g.hour <= onset_hour comparison to behave unexpectedly,#    or patient_id dtype mismatch across merges)print("dtypes:")print(df_check.dtypes)

## 14. COLD START — run this ONE cell after a session disconnectThis replaces needing to hunt through earlier sections. It reloadseverything needed (data, GBT model, threshold, LSTM model + predictions)WITHOUT retraining anything -- just file loads + fast inference passes.**Exact order to run after a disconnect:**1. Cell under Section 1 ("1. Setup") -- installs awscli (needed once per session)2. THIS cell (Section 14)3. Then go straight to Section 12 (lead-time) or Section 13 (diagnostic) --   skip everything in between, it's all rebuilt here.Takes about 1-3 minutes total (data load + GBT inference + LSTM inference,no training).

In [ ]:
import pandas as pd, numpy as np, joblib, torch, torch.nn as nnfrom sklearn.metrics import (roc_auc_score, average_precision_score,                              precision_recall_curve, confusion_matrix)from torch.utils.data import Dataset, DataLoaderfrom torch.nn.utils.rnn import pad_sequence# ---- 1. Mount Drive + set paths ----from google.colab import drivedrive.mount('/content/drive', force_remount=False)DATA_ROOT = '/content/drive/MyDrive/sepsisense_training_data'# ---- 2. Load processed data (already computed earlier, just reading CSVs) ----train_df = pd.read_csv(f"{DATA_ROOT}/../processed_train.csv")val_df = pd.read_csv(f"{DATA_ROOT}/../processed_val.csv")test_df = pd.read_csv(f"{DATA_ROOT}/../processed_test.csv")feature_cols = [c for c in test_df.columns if c not in ("patient_id", "hour", "label")]y_test = test_df["label"].valuesprint(f"Loaded train/val/test: {len(train_df)}/{len(val_df)}/{len(test_df)} rows")# ---- 3. Reload GBT model + recompute its tuned threshold from val set ----gbt = joblib.load(f"{DATA_ROOT}/../gbt_model_full.joblib")test_prob = gbt.predict_proba(test_df[feature_cols].values)[:, 1]val_prob = gbt.predict_proba(val_df[feature_cols].values)[:, 1]precisions, recalls, thresholds = precision_recall_curve(val_df["label"].values, val_prob)f1s = 2 * precisions * recalls / (precisions + recalls + 1e-10)best_thresh = thresholds[np.argmax(f1s[:-1])]print(f"GBT reloaded. Tuned threshold: {best_thresh:.4f}")test_pred = (test_prob >= best_thresh).astype(int)tn, fp, fn, tp = confusion_matrix(y_test, test_pred).ravel()print(f"GBT -> AUROC: {roc_auc_score(y_test, test_prob):.4f}, "      f"AUPRC: {average_precision_score(y_test, test_prob):.4f}, "      f"Precision: {tp/(tp+fp):.4f}, Recall: {tp/(tp+fn):.4f}")# ---- 4. Define LSTM architecture + build test_loader (NO training) ----device = torch.device("cuda" if torch.cuda.is_available() else "cpu")class ICUSequenceDataset(Dataset):    def __init__(self, df, feature_cols):        self.sequences, self.labels = [], []        for pid, g in df.groupby("patient_id"):            g = g.sort_values("hour")            self.sequences.append(torch.tensor(g[feature_cols].values, dtype=torch.float32))            self.labels.append(torch.tensor(g["label"].values, dtype=torch.float32))    def __len__(self): return len(self.sequences)    def __getitem__(self, idx): return self.sequences[idx], self.labels[idx]def collate_fn(batch):    seqs, labels = zip(*batch)    lengths = torch.tensor([len(s) for s in seqs])    return pad_sequence(seqs, batch_first=True), pad_sequence(labels, batch_first=True), lengthsclass SepsisLSTM(nn.Module):    def __init__(self, n_features, hidden_size=64, num_layers=2, dropout=0.3):        super().__init__()        self.lstm = nn.LSTM(n_features, hidden_size, num_layers=num_layers,                             batch_first=True, dropout=dropout, bidirectional=False)        self.fc = nn.Linear(hidden_size, 1)    def forward(self, x, lengths):        packed = nn.utils.rnn.pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)        out, _ = self.lstm(packed)        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True)        return self.fc(out).squeeze(-1)test_ds = ICUSequenceDataset(test_df, feature_cols)test_loader = DataLoader(test_ds, batch_size=64, shuffle=False, collate_fn=collate_fn)n_features = len(feature_cols)model = SepsisLSTM(n_features).to(device)model.load_state_dict(torch.load(f"{DATA_ROOT}/../lstm_model_full.pt", map_location=device))model.eval()print("LSTM reloaded (weights loaded, no training run).")# ---- 5. Run LSTM inference on test set ----all_probs, all_labels = [], []with torch.no_grad():    for seqs, labels, lengths in test_loader:        seqs = seqs.to(device)        logits = model(seqs, lengths)        probs = torch.sigmoid(logits).cpu().numpy()        for i, l in enumerate(lengths):            all_probs.extend(probs[i, :l].tolist())            all_labels.extend(labels[i, :l].tolist())all_probs, all_labels = np.array(all_probs), np.array(all_labels)lstm_precisions, lstm_recalls, lstm_thresholds = precision_recall_curve(all_labels, all_probs)lstm_f1s = 2 * lstm_precisions * lstm_recalls / (lstm_precisions + lstm_recalls + 1e-10)lstm_best_thresh = lstm_thresholds[np.argmax(lstm_f1s[:-1])]print(f"LSTM -> AUROC: {roc_auc_score(all_labels, all_probs):.4f}, "      f"AUPRC: {average_precision_score(all_labels, all_probs):.4f}")print()print("=== COLD START COMPLETE ===")print("Now available: test_df, test_prob, best_thresh (GBT),")print("               all_probs, all_labels (LSTM), gbt, model")print("You can now jump straight to Section 12 (lead-time) or Section 13 (diagnostic).")print("NOTE: LSTM lead-time cell used 'best_thresh' for LSTM alerts too --")print("      if you want LSTM's OWN tuned threshold instead, use lstm_best_thresh")print(f"      (value: {lstm_best_thresh:.4f}) in place of best_thresh in that cell.")

## 15. SHAP Explainability LayerUses SHAP's TreeExplainer on the GBT model (fast, exact for tree models --this is why GBT+SHAP is a very standard combination, more so than LSTM+SHAPwhich needs slower approximate methods). Run after the COLD START cell(Section 14) so `gbt`, `test_df`, `feature_cols`, `test_prob` exist.Produces:  1. A global feature-importance summary (which vitals/labs matter most overall)  2. A per-patient explanation function -- for ANY row, returns the top     contributing factors and their direction (pushing risk up or down)  3. Exports everything the Streamlit dashboard needs to a CSV on Drive

In [ ]:
!pip install shap -qimport shapimport numpy as npimport pandas as pd# TreeExplainer is exact and fast for tree-based models like RandomForest/GBTexplainer = shap.TreeExplainer(gbt)# Computing SHAP values for the full test set can be slow for huge sets --# sample a representative subset (all positives + a random sample of negatives)# for the dashboard demo, which is standard practice for large-scale SHAP use.pos_idx = test_df.index[test_df["label"] == 1].tolist()neg_idx = test_df[test_df["label"] == 0].sample(n=min(2000, (test_df['label']==0).sum()), random_state=42).index.tolist()sample_idx = pos_idx + neg_idxsample_df = test_df.loc[sample_idx].reset_index(drop=True)X_sample = sample_df[feature_cols].valuesshap_values = explainer.shap_values(X_sample)# For binary classification, shap_values may be a list [class0, class1] or a# single array depending on SHAP version -- handle bothif isinstance(shap_values, list):    shap_vals_pos_class = shap_values[1]elif shap_values.ndim == 3:    shap_vals_pos_class = shap_values[:, :, 1]  # (samples, features, classes) -> select positive classelse:    shap_vals_pos_class = shap_valuesprint(f"Computed SHAP values for {len(sample_df)} sample rows "      f"({len(pos_idx)} sepsis-positive + {len(neg_idx)} negative)")

In [ ]:
# ---- Global feature importance (mean absolute SHAP value per feature) ----mean_abs_shap = np.abs(shap_vals_pos_class).mean(axis=0)global_importance = pd.Series(mean_abs_shap, index=feature_cols).sort_values(ascending=False)print("Top 15 globally important features (by mean |SHAP value|):")print(global_importance.head(15))shap.summary_plot(shap_vals_pos_class, X_sample, feature_names=feature_cols, show=True)

In [ ]:
# ---- Per-patient explanation function ----def explain_prediction(row_idx, top_n=5):    '''Returns the top_n features driving this specific prediction, with    direction (positive SHAP = pushes risk UP, negative = pushes risk DOWN).'''    row_shap = shap_vals_pos_class[row_idx]    contributions = pd.Series(row_shap, index=feature_cols).sort_values(        key=lambda x: x.abs(), ascending=False    )    top = contributions.head(top_n)    result = []    for feat, val in top.items():        direction = "increases risk" if val > 0 else "decreases risk"        result.append({"feature": feat, "shap_value": float(val), "direction": direction,                        "patient_value": float(sample_df.iloc[row_idx][feat])})    return result# Example: explain the first sepsis-positive patient's onset-hour predictionexample = explain_prediction(0, top_n=5)print("Example explanation (first sepsis-positive sample row):")for item in example:    print(f"  {item['feature']}: value={item['patient_value']:.2f}, "          f"SHAP={item['shap_value']:.4f} ({item['direction']})")

In [ ]:
# ---- Export everything the dashboard needs ----export_df = sample_df[["patient_id", "hour", "label"]].copy()export_df["gbt_prob"] = gbt.predict_proba(X_sample)[:, 1]# Attach full LSTM predictions where available (matched by patient_id/hour)# -- assumes all_probs/all_labels + the same patient/hour ordering used# earlier in the notebook (Section 11/14). If not available, skip this join.try:    lstm_lookup = df_check[["patient_id", "hour"]].copy()  # from diagnostic cell, if run    lstm_lookup["lstm_prob"] = all_probs    export_df = export_df.merge(lstm_lookup, on=["patient_id", "hour"], how="left")except NameError:    print("LSTM probs not merged (run Section 13 diagnostic cell first if you want this).")# Save per-row SHAP values as a wide CSV (one column per feature)shap_df = pd.DataFrame(shap_vals_pos_class, columns=[f"shap_{c}" for c in feature_cols])export_df = pd.concat([export_df.reset_index(drop=True), shap_df.reset_index(drop=True)], axis=1)# Also attach raw feature values (needed for the dashboard to show actual vitals)raw_vals_df = sample_df[feature_cols].reset_index(drop=True)raw_vals_df.columns = [f"val_{c}" for c in feature_cols]export_df = pd.concat([export_df, raw_vals_df], axis=1)export_path = f"{DATA_ROOT}/../dashboard_export.csv"export_df.to_csv(export_path, index=False)global_importance.to_csv(f"{DATA_ROOT}/../global_feature_importance.csv")print(f"Saved dashboard data to: {export_path}  ({len(export_df)} rows)")print("Download this file and global_feature_importance.csv from Drive,")print("then place them in your local project's data/processed/ folder")print("for the Streamlit dashboard to use.")

## 16. FULL REBUILD — SOFA + comparison table + lead-times, all in ONE cellRun this instead of hunting through Sections 9-12. Only needs `test_df`,`test_prob`, `best_thresh`, `all_probs`, `all_labels`, `y_test`, `DATA_ROOT`from the COLD START cell (Section 14) -- everything else is recomputed herefrom scratch, no stale variables from earlier sessions.

In [ ]:
import glob as glob_moduleimport numpy as npimport pandas as pdfrom sklearn.metrics import (roc_auc_score, average_precision_score,                              confusion_matrix)# ---------------------------------------------------------------------# PART A: SOFA/qSOFA baseline on the full test set# ---------------------------------------------------------------------QSOFA_ALERT_THRESHOLD = 1SOFA_ALERT_THRESHOLD = 2def qsofa_component(row):    score = 0    if pd.notna(row["Resp"]) and row["Resp"] >= 22: score += 1    if pd.notna(row["SBP"]) and row["SBP"] <= 100: score += 1    return scoredef modified_sofa_component(row):    score = 0    p = row.get("Platelets", np.nan)    if pd.notna(p):        if p < 20: score += 4        elif p < 50: score += 3        elif p < 100: score += 2        elif p < 150: score += 1    b = row.get("Bilirubin_total", np.nan)    if pd.notna(b):        if b >= 12: score += 4        elif b >= 6: score += 3        elif b >= 2: score += 2        elif b >= 1.2: score += 1    m = row.get("MAP", np.nan)    if pd.notna(m) and m < 70: score += 1    c = row.get("Creatinine", np.nan)    if pd.notna(c):        if c >= 5: score += 4        elif c >= 3.5: score += 3        elif c >= 2: score += 2        elif c >= 1.2: score += 1    return scoretest_patient_ids = set(test_df["patient_id"].unique().tolist())psv_lookup = {}for fp in glob_module.glob(f"{DATA_ROOT}/**/*.psv", recursive=True):    pid = fp.split("/")[-1].replace(".psv", "")    if pid in test_patient_ids:        psv_lookup[pid] = fpprint(f"Scoring SOFA/qSOFA on {len(psv_lookup)} real held-out test patients...")sofa_lead_times, sofa_missed, sofa_n_sepsis = [], 0, 0tp_hours = fp_hours = tn_hours = fn_hours = 0for pid, fp in psv_lookup.items():    pdf = pd.read_csv(fp, sep="|")    cols_needed = ["Resp", "SBP", "Platelets", "Bilirubin_total", "MAP", "Creatinine"]    pdf[cols_needed] = pdf[cols_needed].ffill()    pdf["qsofa"] = pdf.apply(qsofa_component, axis=1)    pdf["mod_sofa"] = pdf.apply(modified_sofa_component, axis=1)    pdf["alert"] = (pdf["qsofa"] >= QSOFA_ALERT_THRESHOLD) | (pdf["mod_sofa"] >= SOFA_ALERT_THRESHOLD)    onset_idx = pdf.index[pdf["SepsisLabel"] == 1]    onset_hour = int(onset_idx[0]) if len(onset_idx) > 0 else None    if onset_hour is not None:        pdf = pdf.loc[:onset_hour]    is_sepsis = onset_hour is not None    alerts = pdf.index[pdf["alert"]].tolist()    if is_sepsis:        sofa_n_sepsis += 1        pre_onset_alerts = [a for a in alerts if a <= onset_hour]        if pre_onset_alerts:            sofa_lead_times.append(onset_hour - pre_onset_alerts[0])        else:            sofa_missed += 1    for h in range(len(pdf)):        tl = pdf["SepsisLabel"].iloc[h]        pr = pdf["alert"].iloc[h]        if tl == 1 and pr: tp_hours += 1        elif tl == 1 and not pr: fn_hours += 1        elif tl == 0 and pr: fp_hours += 1        else: tn_hours += 1sofa_precision = tp_hours / (tp_hours + fp_hours) if (tp_hours + fp_hours) > 0 else 0sofa_recall = tp_hours / (tp_hours + fn_hours) if (tp_hours + fn_hours) > 0 else 0sofa_avg_lead = float(np.mean(sofa_lead_times)) if sofa_lead_times else Noneprint(f"SOFA -> Precision: {sofa_precision:.4f}, Recall: {sofa_recall:.4f}, "      f"Avg lead-time: {sofa_avg_lead}")print(f"SOFA detected before/at onset: {sofa_n_sepsis - sofa_missed}/{sofa_n_sepsis}")

In [ ]:
# ---------------------------------------------------------------------# PART B: GBT metrics + lead-time# ---------------------------------------------------------------------test_pred = (test_prob >= best_thresh).astype(int)tn, fp, fn, tp = confusion_matrix(y_test, test_pred).ravel()gbt_precision = tp / (tp + fp) if (tp + fp) > 0 else 0gbt_recall = tp / (tp + fn) if (tp + fn) > 0 else 0gbt_auroc = roc_auc_score(y_test, test_prob)gbt_auprc = average_precision_score(y_test, test_prob)gbt_df = test_df[["patient_id", "hour", "label"]].copy()gbt_df["prob"] = test_probgbt_df["alert"] = gbt_df["prob"] >= best_threshgbt_lead_times, gbt_missed, gbt_n_sepsis = [], 0, 0for pid, g in gbt_df.groupby("patient_id"):    g = g.sort_values("hour")    onset_rows = g.index[g["label"] == 1]    if len(onset_rows) == 0:        continue    gbt_n_sepsis += 1    onset_hour = g.loc[onset_rows[0], "hour"]    pre_onset_alerts = g[(g["alert"]) & (g["hour"] <= onset_hour)]    if len(pre_onset_alerts) > 0:        gbt_lead_times.append(onset_hour - pre_onset_alerts["hour"].min())    else:        gbt_missed += 1gbt_avg_lead = float(np.mean(gbt_lead_times)) if gbt_lead_times else Noneprint(f"GBT -> AUROC: {gbt_auroc:.4f}, AUPRC: {gbt_auprc:.4f}, "      f"Precision: {gbt_precision:.4f}, Recall: {gbt_recall:.4f}")print(f"GBT detected before/at onset: {gbt_n_sepsis - gbt_missed}/{gbt_n_sepsis}, "      f"avg lead-time: {gbt_avg_lead}")

In [ ]:
# ---------------------------------------------------------------------# PART C: LSTM metrics + lead-time# ---------------------------------------------------------------------from sklearn.metrics import precision_recall_curvelstm_precisions, lstm_recalls, lstm_thresholds = precision_recall_curve(all_labels, all_probs)lstm_f1s = 2 * lstm_precisions * lstm_recalls / (lstm_precisions + lstm_recalls + 1e-10)lstm_thresh = lstm_thresholds[np.argmax(lstm_f1s[:-1])]lstm_pred = (all_probs >= lstm_thresh).astype(int)tn, fp, fn, tp = confusion_matrix(all_labels, lstm_pred).ravel()lstm_precision = tp / (tp + fp) if (tp + fp) > 0 else 0lstm_recall = tp / (tp + fn) if (tp + fn) > 0 else 0lstm_auroc = roc_auc_score(all_labels, all_probs)lstm_auprc = average_precision_score(all_labels, all_probs)sorted_pids = sorted(test_df["patient_id"].unique())patient_hour_counts = test_df.groupby("patient_id").size().reindex(sorted_pids)lstm_rows = []cursor = 0for pid, n_hours in patient_hour_counts.items():    p_probs = all_probs[cursor: cursor + n_hours]    p_labels = all_labels[cursor: cursor + n_hours]    for h in range(n_hours):        lstm_rows.append({"patient_id": pid, "hour": h, "label": p_labels[h], "prob": p_probs[h]})    cursor += n_hourslstm_df = pd.DataFrame(lstm_rows)lstm_df["alert"] = lstm_df["prob"] >= lstm_threshlstm_lead_times, lstm_missed, lstm_n_sepsis = [], 0, 0for pid, g in lstm_df.groupby("patient_id"):    g = g.sort_values("hour")    onset_rows = g.index[g["label"] == 1]    if len(onset_rows) == 0:        continue    lstm_n_sepsis += 1    onset_hour = g.loc[onset_rows[0], "hour"]    pre_onset_alerts = g[(g["alert"]) & (g["hour"] <= onset_hour)]    if len(pre_onset_alerts) > 0:        lstm_lead_times.append(onset_hour - pre_onset_alerts["hour"].min())    else:        lstm_missed += 1lstm_avg_lead = float(np.mean(lstm_lead_times)) if lstm_lead_times else Noneprint(f"LSTM -> AUROC: {lstm_auroc:.4f}, AUPRC: {lstm_auprc:.4f}, "      f"Precision: {lstm_precision:.4f}, Recall: {lstm_recall:.4f}")print(f"LSTM detected before/at onset: {lstm_n_sepsis - lstm_missed}/{lstm_n_sepsis}, "      f"avg lead-time: {lstm_avg_lead}")

In [ ]:
# ---------------------------------------------------------------------# PART D: Final table, built fresh, no dependency on old variables# ---------------------------------------------------------------------final_comparison = pd.DataFrame({    "Model": ["SOFA/qSOFA (rule-based)", "GBT (ML)", "LSTM (ML)"],    "AUROC": [None, gbt_auroc, lstm_auroc],    "AUPRC": [None, gbt_auprc, lstm_auprc],    "Precision": [sofa_precision, gbt_precision, lstm_precision],    "Recall": [sofa_recall, gbt_recall, lstm_recall],    "Avg_Lead_Time_Hours": [sofa_avg_lead, gbt_avg_lead, lstm_avg_lead],})print(final_comparison)final_comparison.to_csv(f"{DATA_ROOT}/../final_three_way_comparison.csv", index=False)print("\nSaved complete, consistent final_three_way_comparison.csv")

## 17. FIXED dashboard export — sample PATIENTS (full stay), not random rowsThe earlier export (Section 15) sampled individual rows, which breaks thedashboard's Patient Detail trajectory (most patients only had 1 row) andcaused the merge issue behind the identical-risk-score bug in Ward View.This version keeps every hour for a sample of patients instead, andcorrectly attaches BOTH gbt_prob and lstm_prob per row.Run this AFTER the COLD START cell (Section 14) -- needs `test_df`,`feature_cols`, `gbt`, `all_probs`, `all_labels`, `DATA_ROOT`.

In [ ]:
import shapimport numpy as npimport pandas as pd# ---- 1. Select a sample of PATIENTS (not rows), keeping their full stay ----all_pids = test_df["patient_id"].unique()sepsis_pids = test_df[test_df["label"] == 1]["patient_id"].unique()nonsepsis_pids = np.setdiff1d(all_pids, sepsis_pids)rng = np.random.default_rng(42)# take ALL sepsis patients (usually a manageable number) + a random sample of# non-sepsis patients, so the dashboard has a realistic mix to browsen_nonsepsis_sample = min(150, len(nonsepsis_pids))sampled_nonsepsis = rng.choice(nonsepsis_pids, size=n_nonsepsis_sample, replace=False)sampled_pids = set(sepsis_pids) | set(sampled_nonsepsis)sample_df = test_df[test_df["patient_id"].isin(sampled_pids)].sort_values(    ["patient_id", "hour"]).reset_index(drop=True)print(f"Sampled {len(sampled_pids)} full patients "      f"({len(sepsis_pids)} sepsis + {n_nonsepsis_sample} non-sepsis), "      f"{len(sample_df)} total rows (every hour of their stay kept)")

In [ ]:
# ---- 2. GBT predictions on the sampled rows ----X_sample = sample_df[feature_cols].valuessample_df["gbt_prob"] = gbt.predict_proba(X_sample)[:, 1]# ---- 3. LSTM predictions -- pull from the full all_probs/all_labels arrays#          using the SAME patient/hour reconstruction as the lead-time cells,#          then filter down to just our sampled patients ----sorted_pids_full = sorted(test_df["patient_id"].unique())patient_hour_counts = test_df.groupby("patient_id").size().reindex(sorted_pids_full)lstm_prob_lookup = {}cursor = 0for pid, n_hours in patient_hour_counts.items():    p_probs = all_probs[cursor: cursor + n_hours]    for h in range(n_hours):        lstm_prob_lookup[(pid, h)] = p_probs[h]    cursor += n_hourssample_df["lstm_prob"] = sample_df.apply(    lambda r: lstm_prob_lookup.get((r["patient_id"], r["hour"]), np.nan), axis=1)print(f"LSTM probs attached: {sample_df['lstm_prob'].notna().sum()} / {len(sample_df)} rows matched")print(sample_df[["patient_id", "hour", "gbt_prob", "lstm_prob"]].head(10))

In [ ]:
# ---- 4. SHAP values for this sample (for the 'Why this alert?' panel) ----explainer = shap.TreeExplainer(gbt)shap_values = explainer.shap_values(X_sample)if isinstance(shap_values, list):    shap_vals_pos_class = shap_values[1]elif shap_values.ndim == 3:    shap_vals_pos_class = shap_values[:, :, 1]else:    shap_vals_pos_class = shap_valuesmean_abs_shap = np.abs(shap_vals_pos_class).mean(axis=0)global_importance = pd.Series(mean_abs_shap, index=feature_cols).sort_values(ascending=False)print("Top 10 globally important features:")print(global_importance.head(10))

In [ ]:
# ---- 5. Build and save the corrected dashboard export ----export_df = sample_df[["patient_id", "hour", "label", "gbt_prob", "lstm_prob"]].copy()shap_df = pd.DataFrame(shap_vals_pos_class, columns=[f"shap_{c}" for c in feature_cols])export_df = pd.concat([export_df.reset_index(drop=True), shap_df.reset_index(drop=True)], axis=1)raw_vals_df = sample_df[feature_cols].reset_index(drop=True)raw_vals_df.columns = [f"val_{c}" for c in feature_cols]export_df = pd.concat([export_df, raw_vals_df], axis=1)export_path = f"{DATA_ROOT}/../dashboard_export.csv"export_df.to_csv(export_path, index=False)global_importance.to_csv(f"{DATA_ROOT}/../global_feature_importance.csv")print(f"Saved FIXED dashboard export: {export_path} ({len(export_df)} rows, "      f"{export_df['patient_id'].nunique()} patients, each with their full stay)")print("Re-download this file (overwrite the old one) and re-run the dashboard.")

## 18. SAFE SHAP — small sample, chunked, with visible progressThis replaces Section 17's cells 2-4. Much smaller sample (fast enough toactually finish), and processes in small batches so:  (a) you see live progress instead of a silent 12+ minute freeze, and  (b) you CAN actually interrupt it between batches if needed.Run after the COLD START cell (Section 14).

In [ ]:
import timeimport numpy as npimport pandas as pdimport shap# ---- Small, safe sample: cap sepsis patients too, not just non-sepsis ----all_pids = test_df["patient_id"].unique()sepsis_pids_all = test_df[test_df["label"] == 1]["patient_id"].unique()nonsepsis_pids = np.setdiff1d(all_pids, sepsis_pids_all)rng = np.random.default_rng(42)n_sepsis_sample = min(60, len(sepsis_pids_all))     # cap sepsis patients toon_nonsepsis_sample = min(40, len(nonsepsis_pids))sampled_sepsis = rng.choice(sepsis_pids_all, size=n_sepsis_sample, replace=False)sampled_nonsepsis = rng.choice(nonsepsis_pids, size=n_nonsepsis_sample, replace=False)sampled_pids = set(sampled_sepsis) | set(sampled_nonsepsis)sample_df = test_df[test_df["patient_id"].isin(sampled_pids)].sort_values(    ["patient_id", "hour"])# cap to last 24 hours per patient -- small but still shows a real trendsample_df = sample_df.groupby("patient_id").tail(24).reset_index(drop=True)print(f"Sampled {len(sampled_pids)} patients ({n_sepsis_sample} sepsis + "      f"{n_nonsepsis_sample} non-sepsis), {len(sample_df)} total rows "      f"(last 24h per patient) -- this should be fast.")

In [ ]:
X_sample = sample_df[feature_cols].valuessample_df["gbt_prob"] = gbt.predict_proba(X_sample)[:, 1]explainer = shap.TreeExplainer(gbt)# ---- Chunked SHAP computation with progress + interruptibility ----CHUNK_SIZE = 200n_rows = len(X_sample)shap_chunks = []start = time.time()for i in range(0, n_rows, CHUNK_SIZE):    chunk = X_sample[i:i + CHUNK_SIZE]    chunk_shap = explainer.shap_values(chunk, check_additivity=False)    if isinstance(chunk_shap, list):        chunk_shap = chunk_shap[1]    elif chunk_shap.ndim == 3:        chunk_shap = chunk_shap[:, :, 1]    shap_chunks.append(chunk_shap)    elapsed = time.time() - start    done = min(i + CHUNK_SIZE, n_rows)    print(f"  {done}/{n_rows} rows done ({elapsed:.1f}s elapsed)")shap_vals_pos_class = np.vstack(shap_chunks)print(f"\nDone. Total time: {time.time() - start:.1f}s for {n_rows} rows.")

In [ ]:
mean_abs_shap = np.abs(shap_vals_pos_class).mean(axis=0)global_importance = pd.Series(mean_abs_shap, index=feature_cols).sort_values(ascending=False)print("Top 10 globally important features:")print(global_importance.head(10))

In [ ]:
# ---- LSTM probs for this sample ----sorted_pids_full = sorted(test_df["patient_id"].unique())patient_hour_counts = test_df.groupby("patient_id").size().reindex(sorted_pids_full)lstm_prob_lookup = {}cursor = 0for pid, n_hours in patient_hour_counts.items():    p_probs = all_probs[cursor: cursor + n_hours]    for h in range(n_hours):        lstm_prob_lookup[(pid, h)] = p_probs[h]    cursor += n_hourssample_df["lstm_prob"] = sample_df.apply(    lambda r: lstm_prob_lookup.get((r["patient_id"], r["hour"]), np.nan), axis=1)# ---- Build and save export ----export_df = sample_df[["patient_id", "hour", "label", "gbt_prob", "lstm_prob"]].copy()shap_df = pd.DataFrame(shap_vals_pos_class, columns=[f"shap_{c}" for c in feature_cols])export_df = pd.concat([export_df.reset_index(drop=True), shap_df.reset_index(drop=True)], axis=1)raw_vals_df = sample_df[feature_cols].reset_index(drop=True)raw_vals_df.columns = [f"val_{c}" for c in feature_cols]export_df = pd.concat([export_df, raw_vals_df], axis=1)export_path = f"{DATA_ROOT}/../dashboard_export.csv"export_df.to_csv(export_path, index=False)global_importance.to_csv(f"{DATA_ROOT}/../global_feature_importance.csv")print(f"Saved: {export_path} ({len(export_df)} rows, "      f"{export_df['patient_id'].nunique()} patients)")print("Download and replace the old dashboard_export.csv, then restart the dashboard.")